# LeakAudit - 11 - Bootstrap confidence intervals

**Attach uncertainty to every headline number, especially the small samples.**

- The Hong Kong public sample holds only 140 WAV files from about 50 loggers, so a seed-to-seed standard deviation understates the true uncertainty.
- We report percentile bootstrap 95 % intervals for each mean, and paired bootstrap intervals for the two comparisons the paper rests on: the min-std window heuristic against a random window, and each foundation model's collapse against the in-house baseline.
- Run the relevant experiment notebooks first so the raw CSVs exist.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation, sfb
cfg.apply_mpl_style()
def show_ci(df, group_cols, value_col="acc", n_boot=5000):
    out = []
    for key, g in df.groupby(list(group_cols)):
        ci = sfb.bootstrap_ci(g[value_col].to_numpy(), n_boot=n_boot)
        rec = dict(zip(group_cols, key if isinstance(key, tuple) else (key,)))
        rec.update({"mean": ci["mean"], "ci_lo": ci["lo"],
                    "ci_hi": ci["hi"], "n": ci["n"]})
        out.append(rec)
    return pd.DataFrame(out)

## E4 - protocol collapse (D1)

In [3]:
p = cfg.RESULTS / "E4_collapse.csv"
if p.exists():
    d4 = pd.read_csv(p)
    # Group by model as well: pooling four learners into one interval would
    # describe a mixture rather than any single model.
    keys = ["task", "model", "protocol"] if "model" in d4.columns \
        else ["task", "protocol"]
    t4 = show_ci(d4, keys)
    order = {q: i for i, q in enumerate(evaluation.PROTOCOLS)}
    t4 = t4.sort_values(keys,
                        key=lambda c: c.map(order) if c.name == "protocol" else c)
    print(t4.round(4).to_string(index=False))
    t4.to_csv(cfg.RESULTS / "E4_collapse_ci.csv", index=False)
else:
    print("run notebook 03 first")

              task  model         protocol   mean  ci_lo  ci_hi  n
     T5_five_class   gbdt  P1_naive_random 0.8935 0.8924 0.8946 10
     T5_five_class   gbdt  P2_sensor_indep 0.5435 0.5342 0.5536 10
     T5_five_class   gbdt    P3_site_indep 0.4806 0.4238 0.5367 10
     T5_five_class   gbdt P4_site_temporal 0.3272 0.2648 0.3928 10
     T5_five_class logreg  P1_naive_random 0.6277 0.6258 0.6292 10
     T5_five_class logreg  P2_sensor_indep 0.5132 0.5043 0.5226 10
     T5_five_class logreg    P3_site_indep 0.4655 0.4056 0.5231 10
     T5_five_class logreg P4_site_temporal 0.3026 0.2495 0.3575 10
     T5_five_class    mlp  P1_naive_random 0.8934 0.8881 0.8983 10
     T5_five_class    mlp  P2_sensor_indep 0.5137 0.5039 0.5239 10
     T5_five_class    mlp    P3_site_indep 0.4675 0.4085 0.5265 10
     T5_five_class    mlp P4_site_temporal 0.3148 0.2531 0.3788 10
     T5_five_class     rf  P1_naive_random 0.8496 0.8477 0.8516 10
     T5_five_class     rf  P2_sensor_indep 0.5326 0.5201 0.544

## E7 - foundation models (D2, the small sample)

Here the intervals matter most. We also test each model's collapse
(naive minus logger-independent) with a paired bootstrap.

In [4]:
p = cfg.RESULTS / "E7_foundation_raw.csv"
if p.exists():
    e7 = pd.read_csv(p)
    t7 = show_ci(e7, ["model", "protocol"])
    print(t7.round(4).to_string(index=False))
    t7.to_csv(cfg.RESULTS / "E7_foundation_ci.csv", index=False)

    print("\npaired bootstrap on the collapse (naive - logger-independent):")
    out = []
    for m, g in e7.groupby("model"):
        a = g[g.protocol == "P1_naive"].sort_values("seed").acc.to_numpy()
        b = g[g.protocol == "P2_logger_indep"].sort_values("seed").acc.to_numpy()
        if len(a) and len(b):
            r = sfb.bootstrap_paired_diff(a, b)
            out.append(dict(model=m, collapse=r["diff"], ci_lo=r["lo"],
                            ci_hi=r["hi"], significant=r["excludes_zero"]))
    t7c = pd.DataFrame(out).sort_values("collapse", ascending=False)
    print(t7c.round(4).to_string(index=False))
    t7c.to_csv(cfg.RESULTS / "E7_collapse_ci.csv", index=False)
else:
    print("run notebook 07 first")

        model        protocol   mean  ci_lo  ci_hi  n
          AST        P1_naive 0.8619 0.8405 0.8857 10
          AST P2_logger_indep 0.7665 0.6931 0.8334 10
  HUBERT_BASE        P1_naive 0.8500 0.8190 0.8833 10
  HUBERT_BASE P2_logger_indep 0.7885 0.7176 0.8524 10
      Mel2023        P1_naive 0.7333 0.6905 0.7715 10
      Mel2023 P2_logger_indep 0.6745 0.6338 0.7129 10
WAV2VEC2_BASE        P1_naive 0.7857 0.7571 0.8167 10
WAV2VEC2_BASE P2_logger_indep 0.7288 0.6571 0.7928 10

paired bootstrap on the collapse (naive - logger-independent):
        model  collapse   ci_lo  ci_hi  significant
          AST    0.0954  0.0329 0.1612         True
  HUBERT_BASE    0.0615 -0.0108 0.1413        False
      Mel2023    0.0588 -0.0216 0.1356        False
WAV2VEC2_BASE    0.0569 -0.0055 0.1265        False


## E8 - min-std window heuristic (D3)

The paper's self-audit: does the 2023 heuristic beat a random window? A paired
bootstrap interval that excludes zero settles the question; an interval that
spans zero means the two are indistinguishable at this sample size.

Note that this experiment has limited power: 120 recordings with an 80/20 class
split, so every window mode sits within a couple of points of the majority-class
baseline. Read the intervals, not the point estimates.

In [5]:
p = cfg.RESULTS / "E8_window_heuristic_raw.csv"
if p.exists():
    e8 = pd.read_csv(p)
    t8 = show_ci(e8, ["mode", "split"])
    print(t8.round(4).to_string(index=False))
    t8.to_csv(cfg.RESULTS / "E8_window_heuristic_ci.csv", index=False)

    print("\npaired bootstrap, minstd - random:")
    out = []
    for sp in e8.split.unique():
        a = e8[(e8["mode"] == "minstd") & (e8.split == sp)].sort_values("seed").acc.to_numpy()
        b = e8[(e8["mode"] == "random") & (e8.split == sp)].sort_values("seed").acc.to_numpy()
        if len(a) and len(b):
            r = sfb.bootstrap_paired_diff(a, b)
            out.append(dict(split=sp, diff=r["diff"], ci_lo=r["lo"],
                            ci_hi=r["hi"], significant=r["excludes_zero"]))
    t8c = pd.DataFrame(out)
    print(t8c.round(4).to_string(index=False))
    t8c.to_csv(cfg.RESULTS / "E8_minstd_vs_random_ci.csv", index=False)
    # State the verdict from the interval rather than assuming a direction.
    for _, r in t8c.iterrows():
        if not r.significant:
            print(f"\n-> {r.split}: the interval spans zero, so the 2023 heuristic is")
            print("   indistinguishable from a random window in this setting.")
        elif r["diff"] < 0:
            print(f"\n-> {r.split}: the interval is negative and excludes zero, so the")
            print("   2023 heuristic is worse than a random window.")
        else:
            print(f"\n-> {r.split}: the interval is positive and excludes zero, so the")
            print("   2023 heuristic beats a random window.")
else:
    print("run notebook 05 first")

  mode           split   mean  ci_lo  ci_hi  n
 first condition_indep 0.6222 0.5889 0.6556  5
 first           naive 0.7806 0.7417 0.8139 10
  full condition_indep 0.6722 0.6667 0.6833  5
  full           naive 0.8194 0.7889 0.8528 10
minstd condition_indep 0.6167 0.6000 0.6333  5
minstd           naive 0.8083 0.7778 0.8389 10
random condition_indep 0.6333 0.6111 0.6556  5
random           naive 0.8000 0.7694 0.8250 10

paired bootstrap, minstd - random:
          split    diff   ci_lo  ci_hi  significant
          naive  0.0083 -0.0194 0.0333        False
condition_indep -0.0167 -0.0389 0.0056        False

-> naive: the interval spans zero, so the 2023 heuristic is
   indistinguishable from a random window in this setting.

-> condition_indep: the interval spans zero, so the 2023 heuristic is
   indistinguishable from a random window in this setting.


## E5 and E10 - remaining experiments

In [6]:
for name, path, cols in [
    ("E5 recording leakage", "E5_recording_leakage_raw.csv", ["protocol"]),
    ("E10 mitigation",       "E10_mitigation_raw.csv",       ["method"]),
]:
    p = cfg.RESULTS / path
    if p.exists():
        d = pd.read_csv(p)
        t = show_ci(d, cols)
        print(f"\n{name}")
        print(t.round(4).to_string(index=False))
        t.to_csv(cfg.RESULTS / path.replace("_raw", "_ci"), index=False)
    else:
        print(f"\n{name}: not run yet")


E5 recording leakage
          protocol   mean  ci_lo  ci_hi  n
 S1_segment_random 0.9881 0.9867 0.9894 10
S2_recording_indep 0.9052 0.8749 0.9311 10
S3_condition_indep 0.5950 0.4946 0.6600  5

E10 mitigation
         method   mean  ci_lo  ci_hi  n
            ERM 0.5016 0.4799 0.5204  5
GroupDRO_approx 0.5061 0.4883 0.5213  5
            JTT 0.5114 0.4933 0.5295  5
